In [11]:
import torch
import torch.nn as nn 
from torch.optim import Adam 
from torch.utils.data import TensorDataset, Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# tensor de treinamento
X_train = torch.tensor([[[[4,5,6,7], [5,6,7,8], [8,9,10,11], [4,6,7,8]]],[[[-4,5,6,-7], [5,-6,7,8], [-8,9,-10,11], [-4,-6,-7,-8]]]]).float().to(device)
# divide todos os valores por 8
X_train.div_(8)
# resposta
y_train = torch.tensor([0,1]).float().to(device)

In [12]:
X_train.shape

torch.Size([2, 1, 4, 4])

In [7]:
y_train.shape

torch.Size([2])

In [8]:
# função para definir o modelo
def get_model():
    # rede sequencial
    model = nn.Sequential(
        # convolução
        nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3),
        # pooling
        nn.MaxPool2d(kernel_size=2),
        nn.ReLU(), 
        # transforma em 2d
        nn.Flatten(), 
        nn.Linear(1, 1),
        nn.Sigmoid()
    ).to(device)

    # função de perda
    loss_fn = nn.BCELoss()
    # optimizador
    optimizer = Adam(model.parameters(), lr=0.01)

    return model, loss_fn, optimizer

In [9]:
# instancia o modelo
model, criterion, optimizer = get_model()

In [10]:
model

Sequential(
  (0): Conv2d(1, 1, kernel_size=(3, 3), stride=(1, 1))
  (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (2): ReLU()
  (3): Flatten(start_dim=1, end_dim=-1)
  (4): Linear(in_features=1, out_features=1, bias=True)
  (5): Sigmoid()
)

In [13]:
# função com os lotes de treinamento
def train_batch(x, y, model, optimizer, loss_fn):
    model.train()
    optimizer.zero_grad()
    predictions = model(x)
    batch_loss = loss_fn(predictions.squeeze(), y.squeeze())
    batch_loss.backward()
    optimizer.step()
    return batch_loss.item()

In [14]:
# instanciando o data loader
trn_dl = DataLoader(TensorDataset(X_train, y_train))

In [15]:
# treinamento de 2000 épocas
for epoch in range(2000):
    for ix, batch in enumerate(trn_dl):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        batch_loss = train_batch(x, y, model, optimizer, criterion)

In [16]:
model(X_train[:1])

tensor([[0.0038]], grad_fn=<SigmoidBackward0>)